# Assignment 1 — Multimodal Image Classification (Traditional Features)

**Dataset:** MS COCO 2017 (validation set recommended)

This notebook implements a reproducible pipeline for the assignment:
- Canny edge features for images
- Word2Vec embeddings for captions
- Feature fusion (concatenation)
- Classifier: Logistic Regression (unimodal vs multimodal comparison)

### Notes
- The notebook contains code that *assumes* you have the COCO `val2017` images and `annotations/instances_val2017.json` and `captions_val2017.json` locally. 
- If you don't have a pretrained Word2Vec model available, the notebook will train a small Word2Vec on the captions (fast but approximate).
- For large downloads (COCO images, pretrained Word2Vec), follow the commented-downloaded instructions in the cells.


In [1]:
# Setup: imports and helper installs (run once)
# If you need to install missing libraries, uncomment the pip lines below.
# Note: this environment may not have internet access; run pip installs locally if needed.

# !pip install pycocotools opencv-python gensim scikit-learn matplotlib seaborn tqdm pillow

import os
import json
from pathlib import Path
from tqdm import tqdm
import numpy as np
import cv2
from PIL import Image
from gensim.models import Word2Vec, KeyedVectors
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

print('Libraries imported.')

Libraries imported.


## 1) Dataset preparation

Place the COCO validation images and caption annotation file as follows:

```
/path/to/coco/val2017/  # images (e.g., 5000 images)
/path/to/coco/annotations/captions_val2017.json
/path/to/coco/annotations/instances_val2017.json  # if you need labels
```

The assignment asks for image classification across 80 COCO categories. We'll build an image-level label from instance annotations (assign the most frequent object category present in the image).

In [ ]:
# Edit these paths to match your local setup
COCO_ROOT = '/home/BTECH_7TH_SEM/MS-COCO'  # <<-- CHANGE THIS
IM_DIR = os.path.join(COCO_ROOT, 'val2017')
CAPTIONS_JSON = os.path.join(COCO_ROOT,'annotations_trainval2017', 'annotations', 'captions_val2017.json')
INSTANCES_JSON = os.path.join(COCO_ROOT, 'annotations_trainval2017', 'annotations', 'instances_val2017.json')

# --- Utility loader ---
def load_coco_captions(captions_json):
    with open(captions_json, 'r', encoding='utf-8') as f:
        data = json.load(f)
    # Build image_id -> list of captions mapping
    img2caps = {}
    for item in data['annotations']:
        img_id = item['image_id']
        img2caps.setdefault(img_id, []).append(item['caption'])
    return img2caps

def load_image_labels(instances_json):
    # Build image_id -> list of category_ids
    with open(instances_json, 'r', encoding='utf-8') as f:
        data = json.load(f)
    img2cat = {}
    catid2name = {c['id']:c['name'] for c in data['categories']}
    for ann in data['annotations']:
        img_id = ann['image_id']
        cat_id = ann['category_id']
        img2cat.setdefault(img_id, []).append(cat_id)
    # Convert to most frequent category per image (simple heuristic)
    img2label = {}
    for img_id, cats in img2cat.items():
        most = max(set(cats), key=cats.count)
        img2label[img_id] = most
    return img2label, catid2name

# Example usage (edit path above, then run):
# img2caps = load_coco_captions(CAPTIONS_JSON)
# img2label, catid2name = load_image_labels(INSTANCES_JSON)
# print('Example image_id keys:', list(img2caps.keys())[:5])

In [ ]:
# ---------- Feature extraction helpers ----------
import re
def preprocess_caption(c):
    # lowercase, remove non-alpha, simple tokenization
    c = c.lower()
    c = re.sub(r"[^a-z0-9\s]", '', c)
    tokens = c.split()
    return tokens

def extract_canny_features(image_path, resize=(224,224), sigma=1.0):
    # Load image as grayscale, resize, apply Canny, return flattened histogram of edges
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(image_path)
    img = cv2.resize(img, resize)
    # Blur for stability
    blur = cv2.GaussianBlur(img, (5,5), sigma)
    edges = cv2.Canny(blur, 100, 200)
    # compute simple feature: normalized histogram of edge pixels by row/col or flatten
    # Here we'll use a small downsampled edge map as feature (e.g., 28x28)
    small = cv2.resize(edges, (28,28)).astype(np.float32)/255.0
    feat = small.flatten()  # 784-dim
    return feat

def caption_to_embedding(caption_tokens, w2v_model, dim=300, agg='avg'):
    # caption_tokens: list of tokens
    vecs = []
    for t in caption_tokens:
        if t in w2v_model.wv:
            vecs.append(w2v_model.wv[t])
    if len(vecs)==0:
        return np.zeros(dim, dtype=np.float32)
    vecs = np.stack(vecs)
    if agg=='avg':
        return vecs.mean(axis=0)
    elif agg=='max':
        return vecs.max(axis=0)
    else:
        return vecs.mean(axis=0)


In [ ]:
# Build dataset features (this may take time). We'll demonstrate with a small subset.
def build_features(img2caps, img2label, image_dir, max_images=None, w2v_model=None):
    X_img = []
    X_txt = []
    y = []
    img_ids = list(img2caps.keys())
    if max_images:
        img_ids = img_ids[:max_images]
    missing = 0
    for img_id in tqdm(img_ids):
        img_fname = f'{img_id:012d}.jpg'
        img_path = os.path.join(image_dir, img_fname)
        try:
            img_feat = extract_canny_features(img_path)
        except FileNotFoundError:
            missing += 1
            continue
        # captions -> aggregate embedding: average over 5 captions
        caps = img2caps.get(img_id, [])
        cap_embs = []
        for c in caps:
            toks = preprocess_caption(c)
            if w2v_model:
                cap_embs.append(caption_to_embedding(toks, w2v_model))
        if len(cap_embs)==0:
            # fallback zeros
            cap_emb = np.zeros(w2v_model.vector_size if w2v_model else 300)
        else:
            cap_emb = np.stack(cap_embs).mean(axis=0)
        label = img2label.get(img_id, None)
        if label is None:
            continue
        X_img.append(img_feat)
        X_txt.append(cap_emb)
        y.append(label)
    print('Missing images:', missing)
    X_img = np.array(X_img)
    X_txt = np.array(X_txt)
    y = np.array(y)
    return X_img, X_txt, y

# NOTE: For quick experiments, call with max_images=2000 or less.


In [ ]:
# Word2Vec: if you don't have a pretrained model, train a small one on the captions.
def train_w2v_on_captions(img2caps, vector_size=300, window=5, min_count=2, epochs=10):
    sentences = []
    for caps in img2caps.values():
        for c in caps:
            sentences.append(preprocess_caption(c))
    print('Training Word2Vec on', len(sentences), 'sentences...')
    model = Word2Vec(sentences, vector_size=vector_size, window=window, min_count=min_count, epochs=epochs)
    return model

# Example:
# w2v_model = train_w2v_on_captions(img2caps, vector_size=300, epochs=10)


In [ ]:
# Fusion: concatenation
def fuse_concat(X_img, X_txt):
    return np.concatenate([X_img, X_txt], axis=1)

# Small pipeline to train and evaluate logistic regression
def train_eval(X, y, test_size=0.2, val_size=0.1, random_state=42):
    X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=test_size, stratify=y, random_state=random_state)
    relative_val = val_size / (1 - test_size)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=relative_val, stratify=y_temp, random_state=random_state)
    print('Train/Val/Test sizes:', X_train.shape[0], X_val.shape[0], X_test.shape[0])
    clf = LogisticRegression(max_iter=1000, multi_class='multinomial', solver='saga', n_jobs=-1)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    return clf, acc, cm, (X_train, X_val, X_test, y_train, y_val, y_test)

# Example usage (after building X_img, X_txt, y):
# X_fused = fuse_concat(X_img, X_txt)
# clf, acc, cm, splits = train_eval(X_fused, y)
# print('Test accuracy:', acc)


In [ ]:
# Conduct unimodal vs multimodal experiments
def run_experiments(X_img, X_txt, y):
    results = {}
    # Image-only
    clf_i, acc_i, cm_i, splits_i = train_eval(X_img, y)
    results['image_only'] = {'clf':clf_i, 'acc':acc_i, 'cm':cm_i}
    # Text-only
    clf_t, acc_t, cm_t, splits_t = train_eval(X_txt, y)
    results['text_only'] = {'clf':clf_t, 'acc':acc_t, 'cm':cm_t}
    # Multimodal (concat)
    X_fused = fuse_concat(X_img, X_txt)
    clf_m, acc_m, cm_m, splits_m = train_eval(X_fused, y)
    results['multimodal'] = {'clf':clf_m, 'acc':acc_m, 'cm':cm_m}
    return results

# Visualization helper for confusion matrix
def plot_confusion(cm, title='Confusion Matrix', labels=None):
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=False, fmt='d', cmap='Blues')
    plt.title(title)
    if labels:
        plt.xticks(np.arange(len(labels))+0.5, labels, rotation=90)
        plt.yticks(np.arange(len(labels))+0.5, labels, rotation=0)
    plt.show()


In [ ]:
# Save models/results
def save_pickle(obj, path):
    with open(path, 'wb') as f:
        pickle.dump(obj, f)

# Example:
# save_pickle(clf_m, 'multimodal_clf.pkl')


### How to run

1. Edit `COCO_ROOT` to point to your local COCO `val2017` and `annotations` directory.
2. Run the loader cells to build `img2caps` and `img2label`.
3. Train or load a Word2Vec model. For speed, train a small one on captions as shown.
4. Build features with `build_features(img2caps, img2label, IM_DIR, max_images=2000, w2v_model=...)`.
5. Run `run_experiments(X_img, X_txt, y)` and inspect accuracies + confusion matrices.

### Notes on reproducibility and assumptions

- We used a simple heuristic to assign a single object category to each image (most frequent object present). This maps the image to one of the 80 COCO categories.
- Canny features are represented by a 28x28 downsampled binary edge map (784-dim). This is a traditional, fixed-size feature.
- Captions are converted to 300-d Word2Vec embeddings and aggregated by averaging 5 caption embeddings per image.
- Fusion strategy used: concatenation of image and text features. You can experiment with element-wise product or learned fusion.

----
If you'd like, I can also create a concise 2-3 page report summarizing the methods and results (Word/PDF).